# Qugeister - QNN色推定モデル学習

このノートブックでは、QuAic HNN Composerで設計したQNNモデルを学習します。

## ワークフロー
1. QuAic HNN Composerで`config.json`をエクスポート
2. このノートブックで学習を実行
3. `weights.pth`をダウンロードしてQuAicに提出

## 1. 環境セットアップ

In [ ]:
# 必要なライブラリをインストール
!pip install -q pennylane torch numpy tqdm

In [ ]:
import json
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pennylane as qml
from tqdm.auto import tqdm
from google.colab import files

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 設定ファイルのアップロード

QuAic HNN Composerからエクスポートした`config.json`をアップロードしてください。

In [ ]:
# config.jsonをアップロード
print('config.jsonをアップロードしてください...')
uploaded = files.upload()

# 設定を読み込み
config_filename = list(uploaded.keys())[0]
with open(config_filename, 'r') as f:
    config = json.load(f)

print(f'\n設定を読み込みました: {config.get("model_name", "unnamed")}')
print(f'説明: {config.get("description", "N/A")}')

## 3. 棋譜データのアップロード

QuAicの「コンペティション > データ」から棋譜データをダウンロードしてアップロードしてください。

In [ ]:
# 棋譜データをアップロード
print('棋譜データ(.pkl)をアップロードしてください...')
uploaded_data = files.upload()
TRAJECTORY_FILE = list(uploaded_data.keys())[0]

# データを読み込み
with open(TRAJECTORY_FILE, 'rb') as f:
    trajectory_data = pickle.load(f)

print(f'読み込んだ試合数: {len(trajectory_data)}')

## 4. QNNモデルの構築

**重要**: このモデルはQuAicの`ExplicitColorEstimationQNN`と互換性のある構造です。

In [ ]:
def parse_config(config):
    """config.jsonから量子回路パラメータを取得"""
    network = config.get('network', {})
    nodes = network.get('nodes', [])

    # 量子ノードを探す
    quantum_nodes = [n for n in nodes if n.get('type') == 'quantum']

    if quantum_nodes:
        q_node = quantum_nodes[0]
        n_qubits = q_node.get('data', {}).get('n_qubits', 4)
        q_circuit = q_node.get('data', {}).get('qCircuit', {})
        n_layers = q_circuit.get('n_layers', 2)
    else:
        n_qubits = 4
        n_layers = 2

    print(f'量子ビット数: {n_qubits}')
    print(f'量子レイヤー数: {n_layers}')

    return n_qubits, n_layers

n_qubits, n_layers = parse_config(config)

In [ ]:
class QuantumLayer(nn.Module):
    """QuAic互換の量子レイヤー
    
    パラメータ構造: weights [n_layers, n_qubits, 2] (RY, RZ)
    """

    def __init__(self, n_qubits, n_layers):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # 量子デバイス (backpropで高速)
        self.dev = qml.device('default.qubit', wires=n_qubits)

        # 量子回路の重み [n_layers, n_qubits, 2] - QuAicと同じ構造
        self.weights = nn.Parameter(
            torch.randn(n_layers, n_qubits, 2) * 0.1
        )

        # 量子回路を定義
        @qml.qnode(self.dev, interface='torch', diff_method='backprop')
        def circuit(inputs, weights):
            # 入力埋め込み (AngleEmbedding - RY)
            for i in range(n_qubits):
                qml.RY(inputs[i], wires=i)

            # Variationalレイヤー
            for layer in range(n_layers):
                # エンタングル
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])

                # パラメータ付き回転 (RY, RZ)
                for i in range(n_qubits):
                    qml.RY(weights[layer, i, 0], wires=i)
                    qml.RZ(weights[layer, i, 1], wires=i)

            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.circuit = circuit

    def forward(self, x):
        batch_size = x.shape[0]
        results = []
        for i in range(batch_size):
            result = self.circuit(x[i], self.weights)
            results.append(torch.stack(result))
        return torch.stack(results)

In [ ]:
class ExplicitColorEstimationQNN(nn.Module):
    """QuAic互換のQNN色推定モデル

    入力: 448次元 (7チャネル × 64)
    出力: [8, 2] (8駒 × good/bad確率)

    構造:
    - preprocessing: 448 -> 128 -> 64 -> n_qubits
    - quantum_layer: n_qubits -> n_qubits
    - color_head: n_qubits -> 64 -> 16
    """

    def __init__(self, n_qubits=4, n_layers=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # 前処理層 (QuAicと同じ構造)
        self.preprocessing = nn.Sequential(
            nn.Linear(448, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, n_qubits),
            nn.Tanh()  # [-1, 1]に制限
        )

        # 量子レイヤー
        self.quantum_layer = QuantumLayer(n_qubits, n_layers)

        # 色推定ヘッド (QuAicと同じ構造)
        self.color_head = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(),
            nn.Linear(64, 16)  # 8駒 × 2 (good/bad)
        )

    def forward(self, x):
        batch_size = x.shape[0]

        # 前処理
        x = self.preprocessing(x)

        # 量子層
        x = self.quantum_layer(x)

        # 色推定
        x = self.color_head(x)  # [batch, 16]
        x = x.view(batch_size, 8, 2)  # [batch, 8, 2]

        return x


# モデルを作成
model = ExplicitColorEstimationQNN(n_qubits=n_qubits, n_layers=n_layers).to(device)
print(f'\nモデルを作成しました')
print(f'パラメータ数: {sum(p.numel() for p in model.parameters()):,}')
print(f'\nstate_dict キー:')
for key in model.state_dict().keys():
    print(f'  {key}')

## 5. データの準備

In [ ]:
def prepare_data(trajectory_data, train_ratio=0.8):
    """棋譜データから学習用データを準備"""
    X_list = []
    y_list = []

    for traj in trajectory_data:
        # Player A
        if 'states_A' in traj and 'true_colors_A' in traj:
            for state, colors in zip(traj['states_A'], traj['true_colors_A']):
                state = np.array(state).flatten()
                colors = np.array(colors)
                if state.shape[0] == 448 and colors.shape[0] == 8:
                    X_list.append(state)
                    y_list.append(colors)

        # Player B
        if 'states_B' in traj and 'true_colors_B' in traj:
            for state, colors in zip(traj['states_B'], traj['true_colors_B']):
                state = np.array(state).flatten()
                colors = np.array(colors)
                if state.shape[0] == 448 and colors.shape[0] == 8:
                    X_list.append(state)
                    y_list.append(colors)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)

    # シャッフル
    indices = np.random.permutation(len(X))
    X, y = X[indices], y[indices]

    # 分割
    split_idx = int(len(X) * train_ratio)
    X_train, X_val = X[:split_idx], X[split_idx:]
    y_train, y_val = y[:split_idx], y[split_idx:]

    print(f'学習データ: {len(X_train)} サンプル')
    print(f'検証データ: {len(X_val)} サンプル')

    return X_train, y_train, X_val, y_val

X_train, y_train, X_val, y_val = prepare_data(trajectory_data)

# DataLoader作成
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train)
)
val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val)
)

batch_size = config.get('training', {}).get('batch_size', 32)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

## 6. 学習

In [ ]:
# 学習設定
training_config = config.get('training', {})
epochs = training_config.get('epochs', 50)
learning_rate = training_config.get('learning_rate', 0.001)

print(f'エポック数: {epochs}')
print(f'学習率: {learning_rate}')
print(f'バッチサイズ: {batch_size}')

# オプティマイザと損失関数
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)  # [batch, 8, 2]

        # 各駒の損失を計算
        loss = criterion(outputs.view(-1, 2), y_batch.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 精度計算
        preds = outputs.argmax(dim=-1)  # [batch, 8]
        correct += (preds == y_batch).sum().item()
        total += y_batch.numel()

    return total_loss / len(loader), correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs.view(-1, 2), y_batch.view(-1))

            total_loss += loss.item()

            preds = outputs.argmax(dim=-1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.numel()

    return total_loss / len(loader), correct / total

In [ ]:
# 学習ループ
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('学習を開始します...\n')

for epoch in tqdm(range(epochs), desc='Training'):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')

print(f'\n学習完了! ベスト検証精度: {best_val_acc:.4f}')

## 7. 学習曲線の可視化

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 損失
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

# 精度
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 8. モデルのエクスポート

学習済みモデルをQuAic形式に変換してダウンロードします。

In [ ]:
def convert_to_quaic_format(state_dict):
    """Colabモデルのstate_dictをQuAic形式に変換

    QuAic expects:
    - preprocessing.* (448 -> n_qubits)
    - quantum_layer.quantum_layer.weights [n_layers, n_qubits, 2]
    - color_head.* (n_qubits -> 16)
    """
    quaic_state = {}

    for key, value in state_dict.items():
        # quantum_layer.weights -> quantum_layer.quantum_layer.weights
        if key == 'quantum_layer.weights':
            quaic_state['quantum_layer.quantum_layer.weights'] = value
        else:
            quaic_state[key] = value

    return quaic_state


# ベストモデルを読み込み
model.load_state_dict(torch.load('best_model.pth'))

# QuAic形式に変換
quaic_state_dict = convert_to_quaic_format(model.state_dict())

# エクスポート
model_name = config.get('model_name', 'qnn_model')
export_filename = f'{model_name}_weights.pth'

torch.save(quaic_state_dict, export_filename)
print(f'モデルを保存しました: {export_filename}')
print(f'\nstate_dict キー:')
for key in quaic_state_dict.keys():
    print(f'  {key}')

In [ ]:
# ダウンロード
print('ダウンロードを開始...')
files.download(export_filename)
print('\nダウンロード完了!')
print('\n次のステップ:')
print('1. QuAic (https://quaic.up.railway.app) にアクセス')
print('2. 「コンペティション > モデル」に移動')
print('3. ダウンロードした .pth ファイルをアップロード')